Install torchinfo for model summary

In [4]:
!pip install torchinfo

In [5]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from transformers import SwinForImageClassification, SwinConfig
from torchinfo import summary
import time
import matplotlib.pyplot as plt
import numpy as np
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Problem 1

In [ ]:
# Hyperparameters
batch_size = 64
image_size = 32
num_classes = 100
learning_rate = 0.001
num_epochs = 10

# Data resizing
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# CIFAR-100 dataset
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                           download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                          download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Patch embedding layer
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                            kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # [B, embed_dim, H', W']
        x = x.flatten(2)  # [B, embed_dim, num_patches]
        x = x.transpose(1, 2)  # [B, num_patches, embed_dim]
        return x

# Transformer Encoder
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x2 = self.layer_norm1(x)
        attention_output, _ = self.attention(x2, x2, x2)
        x = x + attention_output
        x2 = self.layer_norm2(x)
        mlp_output = self.mlp(x2)
        x = x + mlp_output
        return x

# Vision Transformer
class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, embed_dim,
                 num_heads, num_layers, mlp_factor, dropout=0.1):
        super().__init__()
        mlp_dim = embed_dim * mlp_factor
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        num_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.ModuleList(
            [TransformerEncoder(embed_dim, num_heads, mlp_dim, dropout)
             for _ in range(num_layers)]
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        for transformer in self.transformer:
            x = transformer(x)

        x = self.layer_norm(x)
        cls_token_final = x[:, 0]
        x = self.head(cls_token_final)
        return x

100%|██████████| 169M/169M [1:15:41<00:00, 37.2kB/s]


In [8]:
def train_and_evaluate(model, train_loader, test_loader, epochs, lr, model_name="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"\nTraining {model_name}...")
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_start = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            # HF Swin models return a structured output, PyTorch models return logits
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_time = time.time() - epoch_start
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time

    # Testing
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"{model_name} Final Test Accuracy: {accuracy:.2f}% | Total Training Time: {total_time:.2f}s")
    return accuracy, total_time

## Model 1-3

In [6]:
'''
Model 1:
Patch Size: 4x4 , embedded dimension: 256, tranformer blocks 4, attention heads 4, MLP= Embedded* 4
Model 2:
Patch Size: 8x8 , embedded dimension: 512, tranformer blocks 8, attention heads 8, MLP= Embedded*8
Model 3:
Patch Size: 4x4 , embedded dimension: 512, tranformer bocks 4, attention heads 4, MLP= Embedded* 4
'''
image_size = 32
num_classes = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 10
input_size=(batch_size, 3, image_size, image_size)

model_1 = [4, 256, 4, 4, 4]
model_2 = [8, 512, 8, 8, 8]
model_3 = [4, 512, 4, 4, 4]
models = [model_1, model_2, model_3]
for model in models:
    patch_size, embed_dim, num_layers, num_heads, mlp_dim = model
    vit_model = VisionTransformer(patch_size=patch_size, embed_dim=embed_dim, num_layers=num_layers, num_heads=num_heads, mlp_factor = mlp_dim, image_size=image_size, num_classes=num_classes).to(device)
    print(f"\n[ViT Configuration (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim}) Summary]")
    print(summary(vit_model, input_size=input_size))
    train_and_evaluate(vit_model, train_loader, test_loader, num_epochs, learning_rate, f"ViT (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim})")


[ViT Configuration (Patch:4, Dim:256, Layers:4, Heads:4, MLP:4) Summary]
Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [64, 100]                 16,896
├─PatchEmbedding: 1-1                    [64, 64, 256]             --
│    └─Conv2d: 2-1                       [64, 256, 8, 8]           12,544
├─Dropout: 1-2                           [64, 65, 256]             --
├─ModuleList: 1-3                        --                        --
│    └─TransformerEncoder: 2-2           [64, 65, 256]             --
│    │    └─LayerNorm: 3-1               [64, 65, 256]             512
│    │    └─MultiheadAttention: 3-2      [64, 65, 256]             263,168
│    │    └─LayerNorm: 3-3               [64, 65, 256]             512
│    │    └─Sequential: 3-4              [64, 65, 256]             525,568
│    └─TransformerEncoder: 2-3           [64, 65, 256]             --
│    │    └─LayerNorm: 3-5               [64, 65, 256]       

## Model 4-6

In [9]:
'''
Model 4:
Patch Size: 4x4 , embedded dimension: 128, tranformer bocks 4, attention heads 4, MLP= Embedded* 4
Model 5:
Patch Size: 8x8 , embedded dimension: 256, tranformer bocks 4, attention heads 4, MLP= Embedded* 4
Model 6:
Patch Size: 8x8 , embedded dimension: 256, tranformer bocks 4, attention heads 4, MLP= Embedded* 8
'''
image_size = 32
num_classes = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 10
input_size=(batch_size, 3, image_size, image_size)


model_4 = [4, 128, 4, 4, 4]
model_5 = [8, 256, 4, 4, 4]
model_6 = [8, 256, 4, 4, 8]
models = [model_4, model_5, model_6]
for model in models:
    patch_size, embed_dim, num_layers, num_heads, mlp_dim = model
    vit_model = VisionTransformer(patch_size=patch_size, embed_dim=embed_dim, num_layers=num_layers, num_heads=num_heads, mlp_factor = mlp_dim, image_size=image_size, num_classes=num_classes).to(device)
    print(f"\n[ViT Configuration (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim}) Summary]")
    print(summary(vit_model, input_size=input_size))
    train_and_evaluate(vit_model, train_loader, test_loader, num_epochs, learning_rate, f"ViT (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim})")


[ViT Configuration (Patch:4, Dim:128, Layers:4, Heads:4, MLP:4) Summary]
Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [64, 100]                 8,448
├─PatchEmbedding: 1-1                    [64, 64, 128]             --
│    └─Conv2d: 2-1                       [64, 128, 8, 8]           6,272
├─Dropout: 1-2                           [64, 65, 128]             --
├─ModuleList: 1-3                        --                        --
│    └─TransformerEncoder: 2-2           [64, 65, 128]             --
│    │    └─LayerNorm: 3-1               [64, 65, 128]             256
│    │    └─MultiheadAttention: 3-2      [64, 65, 128]             66,048
│    │    └─LayerNorm: 3-3               [64, 65, 128]             256
│    │    └─Sequential: 3-4              [64, 65, 128]             131,712
│    └─TransformerEncoder: 2-3           [64, 65, 128]             --
│    │    └─LayerNorm: 3-5               [64, 65, 128]          

## ResNet-18

In [10]:
# 2. ResNet-18 Baseline
resnet = resnet18(weights=None) # From scratch as baseline, or use ResNet18_Weights.DEFAULT
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes) # Adjust for CIFAR-100
resnet = resnet.to(device)
print("\n[ResNet-18 Summary]")
summary(resnet, input_size=(batch_size, 3, image_size, image_size))
train_and_evaluate(resnet, train_loader, test_loader, num_epochs, learning_rate, "ResNet-18 Baseline")


[ResNet-18 Summary]

Training ResNet-18 Baseline...
Epoch [1/10] | Loss: 3.5529 | Time: 20.30s
Epoch [2/10] | Loss: 2.8256 | Time: 20.17s
Epoch [3/10] | Loss: 2.4212 | Time: 20.19s
Epoch [4/10] | Loss: 2.1221 | Time: 20.17s
Epoch [5/10] | Loss: 1.8643 | Time: 19.98s
Epoch [6/10] | Loss: 1.6152 | Time: 20.20s
Epoch [7/10] | Loss: 1.3629 | Time: 20.11s
Epoch [8/10] | Loss: 1.1071 | Time: 20.08s
Epoch [9/10] | Loss: 0.8727 | Time: 20.02s
Epoch [10/10] | Loss: 0.6662 | Time: 20.08s
ResNet-18 Baseline Final Test Accuracy: 44.35% | Total Training Time: 201.31s


(44.35, 201.31434535980225)

## Model 7

In [11]:
image_size = 32
num_classes = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 10
input_size=(batch_size, 3, image_size, image_size)


model_7 = [4, 64, 4, 4, 4]
models = [model_7]
for model in models:
    patch_size, embed_dim, num_layers, num_heads, mlp_dim = model
    vit_model = VisionTransformer(patch_size=patch_size, embed_dim=embed_dim, num_layers=num_layers, num_heads=num_heads, mlp_factor = mlp_dim, image_size=image_size, num_classes=num_classes).to(device)
    print(f"\n[ViT Configuration (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim}) Summary]")
    print(summary(vit_model, input_size=input_size))
    train_and_evaluate(vit_model, train_loader, test_loader, num_epochs, learning_rate, f"ViT (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim})")


[ViT Configuration (Patch:4, Dim:64, Layers:4, Heads:4, MLP:4) Summary]
Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [64, 100]                 4,224
├─PatchEmbedding: 1-1                    [64, 64, 64]              --
│    └─Conv2d: 2-1                       [64, 64, 8, 8]            3,136
├─Dropout: 1-2                           [64, 65, 64]              --
├─ModuleList: 1-3                        --                        --
│    └─TransformerEncoder: 2-2           [64, 65, 64]              --
│    │    └─LayerNorm: 3-1               [64, 65, 64]              128
│    │    └─MultiheadAttention: 3-2      [64, 65, 64]              16,640
│    │    └─LayerNorm: 3-3               [64, 65, 64]              128
│    │    └─Sequential: 3-4              [64, 65, 64]              33,088
│    └─TransformerEncoder: 2-3           [64, 65, 64]              --
│    │    └─LayerNorm: 3-5               [64, 65, 64]             

## Model 7

In [12]:
image_size = 32
num_classes = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 10
input_size=(batch_size, 3, image_size, image_size)


model_7 = [4, 64, 4, 8, 4]
models = [model_7]
for model in models:
    patch_size, embed_dim, num_layers, num_heads, mlp_dim = model
    vit_model = VisionTransformer(patch_size=patch_size, embed_dim=embed_dim, num_layers=num_layers, num_heads=num_heads, mlp_factor = mlp_dim, image_size=image_size, num_classes=num_classes).to(device)
    print(f"\n[ViT Configuration (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim}) Summary]")
    print(summary(vit_model, input_size=input_size))
    train_and_evaluate(vit_model, train_loader, test_loader, num_epochs, learning_rate, f"ViT (Patch:{patch_size}, Dim:{embed_dim}, Layers:{num_layers}, Heads:{num_heads}, MLP:{mlp_dim})")


[ViT Configuration (Patch:4, Dim:64, Layers:4, Heads:8, MLP:4) Summary]
Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [64, 100]                 4,224
├─PatchEmbedding: 1-1                    [64, 64, 64]              --
│    └─Conv2d: 2-1                       [64, 64, 8, 8]            3,136
├─Dropout: 1-2                           [64, 65, 64]              --
├─ModuleList: 1-3                        --                        --
│    └─TransformerEncoder: 2-2           [64, 65, 64]              --
│    │    └─LayerNorm: 3-1               [64, 65, 64]              128
│    │    └─MultiheadAttention: 3-2      [64, 65, 64]              16,640
│    │    └─LayerNorm: 3-3               [64, 65, 64]              128
│    │    └─Sequential: 3-4              [64, 65, 64]              33,088
│    └─TransformerEncoder: 2-3           [64, 65, 64]              --
│    │    └─LayerNorm: 3-5               [64, 65, 64]             

# Problem 2

In [13]:
batch_size = 32
num_epochs = 5
lr_pretrained = 2e-5
lr_scratch = 0.001
num_classes = 100

# Transforms & Data (Swin requires 224x224 inputs)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

def prepare_swin(model_name):
    """Loads a Swin Transformer, freezes backbone if pretrained, and replaces head."""
    model = SwinForImageClassification.from_pretrained(model_name, ignore_mismatched_sizes=True)
    # Freeze backbone
    for param in model.swin.parameters():
        param.requires_grad = False

    # Replace classifier head for 100 classes
    for param in model.classifier.parameters():
      param.requires_grad = True
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    return model.to(device)

## Swin Pretrained

In [15]:
input_size = (batch_size, 3, 224, 224)

# 1. Swin-Tiny (Pretrained)
print("\nLoading Pretrained Swin-Tiny...")
swin_tiny = prepare_swin("microsoft/swin-tiny-patch4-window7-224")
print("\n[Swin-Tiny Summary]")
summary(swin_tiny, input_size=input_size)
train_and_evaluate(swin_tiny, train_loader, test_loader, num_epochs, lr_pretrained, "Swin-Tiny (Pretrained)")

# 2. Swin-Small (Pretrained)
print("\nLoading Pretrained Swin-Small...")
swin_small = prepare_swin("microsoft/swin-small-patch4-window7-224")
train_and_evaluate(swin_small, train_loader, test_loader, num_epochs, lr_pretrained, "Swin-Small (Pretrained)")

# 3. Swin-Tiny (Custom from Scratch)
print("\nLoading Custom Swin-Tiny from Scratch...")


Loading Pretrained Swin-Tiny...


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]


[Swin-Tiny Summary]

Training Swin-Tiny (Pretrained)...
Epoch [1/5] | Loss: 4.0385 | Time: 105.08s
Epoch [2/5] | Loss: 3.0460 | Time: 105.28s
Epoch [3/5] | Loss: 2.3705 | Time: 105.11s
Epoch [4/5] | Loss: 1.9411 | Time: 105.21s
Epoch [5/5] | Loss: 1.6710 | Time: 105.14s
Swin-Tiny (Pretrained) Final Test Accuracy: 66.27% | Total Training Time: 525.82s

Loading Pretrained Swin-Small...


...

Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]


Training Swin-Small (Pretrained)...


model.safetensors:   0%|          | 0.00/199M [00:00<?, ?B/s]

Epoch [1/5] | Loss: 3.9855 | Time: 140.04s
Epoch [2/5] | Loss: 2.8922 | Time: 139.56s
Epoch [3/5] | Loss: 2.1690 | Time: 139.93s
Epoch [4/5] | Loss: 1.7350 | Time: 139.54s
Epoch [5/5] | Loss: 1.4740 | Time: 139.96s
Swin-Small (Pretrained) Final Test Accuracy: 69.93% | Total Training Time: 699.03s

Loading Custom Swin-Tiny from Scratch...

[Custom Swin-Tiny Summary]

Training Swin-Tiny (Custom from Scratch)...
Epoch [1/5] | Loss: 4.6305 | Time: 200.92s
Epoch [2/5] | Loss: 4.6223 | Time: 200.44s
Epoch [3/5] | Loss: 4.6141 | Time: 200.63s
Epoch [4/5] | Loss: 4.6112 | Time: 200.43s
Epoch [5/5] | Loss: 4.6103 | Time: 200.63s
Swin-Tiny (Custom from Scratch) Final Test Accuracy: 1.00% | Total Training Time: 1003.05s


(1.0, 1003.0491533279419)

## Swin from Scratch

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. Helper Functions
# ==========================================
def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

# ==========================================
# 2. Window Attention with Relative Position Bias
# ==========================================
class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size  # (Wh, Ww)
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Define relative position bias table
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))

        # Get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        # Add relative position bias
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        # Apply cyclic shift mask if provided
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x

# ==========================================
# 3. Swin Transformer Block (Handles W-MSA and SW-MSA)
# ==========================================
class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size=(self.window_size, self.window_size), num_heads=num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

        # Calculate attention mask for SW-MSA
        if self.shift_size > 0:
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            mask_windows = window_partition(img_mask, self.window_size)
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
            self.register_buffer("attn_mask", attn_mask)
        else:
            self.attn_mask = None

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # Cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # Partition windows
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)

        # W-MSA / SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)

        # Merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)

        # Reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

# ==========================================
# 4. Patch Merging (Downsampling)
# ==========================================
class PatchMerging(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)

        x = self.norm(x)
        x = self.reduction(x)
        return x

# ==========================================
# 5. Basic Layer (A full stage of blocks)
# ==========================================
class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size, downsample=None):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, input_resolution=input_resolution, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2)
            for i in range(depth)])

        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim)
        else:
            self.downsample = None

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

# ==========================================
# 6. Full Authentic Swin Transformer
# ==========================================
class AuthenticSwinTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=100,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24], window_size=7):
        super().__init__()

        # Patch Embedding
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        patches_resolution = [img_size // patch_size, img_size // patch_size]

        self.pos_drop = nn.Dropout(p=0.1)
        self.layers = nn.ModuleList()

        # Build stages
        for i_layer in range(len(depths)):
            layer = BasicLayer(
                dim=int(embed_dim * 2 ** i_layer),
                input_resolution=(patches_resolution[0] // (2 ** i_layer), patches_resolution[1] // (2 ** i_layer)),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                downsample=PatchMerging if (i_layer < len(depths) - 1) else None)
            self.layers.append(layer)

        self.norm = nn.LayerNorm(int(embed_dim * 2 ** (len(depths) - 1)))
        self.head = nn.Linear(int(embed_dim * 2 ** (len(depths) - 1)), num_classes)

    def forward(self, x):
        # [B, C, H, W] -> [B, Embed_Dim, H/4, W/4]
        x = self.patch_embed(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # [B, H*W, C]
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.head(x)

In [18]:
# Initialize authentic Swin-Tiny
swin_scratch = AuthenticSwinTransformer(
    img_size=224,
    patch_size=4,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
    window_size=7,
    num_classes=100
).to(device)

# Train your authentic implementation
train_and_evaluate(swin_scratch, train_loader, test_loader, num_epochs, lr_scratch, "Authentic Swin-Tiny (Scratch)")


[Authentic Swin-Tiny Summary]

Training Authentic Swin-Tiny (Scratch)...
Epoch [1/5] | Loss: 4.6566 | Time: 205.07s
Epoch [2/5] | Loss: 4.6207 | Time: 205.35s
Epoch [3/5] | Loss: 4.6136 | Time: 207.32s
Epoch [4/5] | Loss: 4.6109 | Time: 208.13s
Epoch [5/5] | Loss: 4.6098 | Time: 208.15s
Authentic Swin-Tiny (Scratch) Final Test Accuracy: 1.00% | Total Training Time: 1034.02s


(1.0, 1034.0239651203156)

In [3]:
def train_and_evaluate(model, train_loader, test_loader, epochs, lr, model_name="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"\nTraining {model_name}...")
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_start = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            # HF Swin models return a structured output, PyTorch models return logits
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_time = time.time() - epoch_start
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time

    # Testing
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"{model_name} Final Test Accuracy: {accuracy:.2f}% | Total Training Time: {total_time:.2f}s")
    return accuracy, total_time
batch_size = 32
num_epochs = 5
lr_pretrained = 2e-5
lr_scratch = 0.001
num_classes = 100

# Transforms & Data (Swin requires 224x224 inputs)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def prepare_swin(model_name, pretrained=True):
    """Loads a Swin Transformer, freezes backbone if pretrained, and replaces head."""
    if pretrained:
        model = SwinForImageClassification.from_pretrained(model_name, ignore_mismatched_sizes=True)
        # Freeze backbone
        for param in model.swin.parameters():
            param.requires_grad = False
    else:
        # Load from scratch (random weights)
        config = SwinConfig.from_pretrained(model_name, num_labels=num_classes)
        model = SwinForImageClassification(config)

    # Replace classifier head for 100 classes
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model.to(device)

print("\nLoading Swin-Tiny from Scratch...")
swin_scratch = prepare_swin("microsoft/swin-tiny-patch4-window7-224", pretrained=False)
train_and_evaluate(swin_scratch, train_loader, test_loader, num_epochs, lr_scratch, "Swin-Tiny (Scratch)")

100%|██████████| 169M/169M [54:53<00:00, 51.3kB/s]



Loading Swin-Tiny from Scratch...


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


NameError: name 'device' is not defined

## Swin Without Pretraining

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def prepare_swin(model_name, pretrained=True):
    """Loads a Swin Transformer, freezes backbone if pretrained, and replaces head."""
    if pretrained:
        model = SwinForImageClassification.from_pretrained(model_name, ignore_mismatched_sizes=True)
        # Freeze backbone
        for param in model.swin.parameters():
            param.requires_grad = False
    else:
        # Load from scratch (random weights)
        config = SwinConfig.from_pretrained(model_name, num_labels=num_classes)
        model = SwinForImageClassification(config)

    # Replace classifier head for 100 classes
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model.to(device)

print("\nLoading Swin-Tiny from Scratch...")
swin_scratch = prepare_swin("microsoft/swin-tiny-patch4-window7-224", pretrained=False)
train_and_evaluate(swin_scratch, train_loader, test_loader, num_epochs, lr_scratch, "Swin-Tiny (Scratch)")


Loading Swin-Tiny from Scratch...


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.



Training Swin-Tiny (Scratch)...
Epoch [1/5] | Loss: 4.6334 | Time: 193.60s
Epoch [2/5] | Loss: 4.5736 | Time: 191.58s
Epoch [3/5] | Loss: 4.4696 | Time: 191.50s
Epoch [4/5] | Loss: 4.5434 | Time: 191.29s
Epoch [5/5] | Loss: 4.4761 | Time: 191.50s
Swin-Tiny (Scratch) Final Test Accuracy: 3.38% | Total Training Time: 959.48s


(3.38, 959.4762444496155)